# 📋 AI 契約條款自動審查教學
## 用大語言模型幫 B 公司法務部門檢查契約

---

### 🎯 學習目標
透過這份教材，你將學會：
1. 從網路自動下載契約範本與當事人資料
2. 呼叫 Claude API，依照 5 條審查規則逐條分析契約
3. 輸出一份完整的 **契約審查分析報告**

---

### 📂 情境說明
> A 公司提供了一份「技術服務契約」範本給 B 公司。  
> B 公司法務部門需要依照公司內部的 5 條審查規則，逐一檢查這份契約是否有問題。  
> 過去需要人工逐條閱讀，現在我們用 AI 來自動化這個流程！

---

### 🔄 執行流程圖

```
📁 party_data.json          📄 contract_template.txt
（GitHub 上的當事人資料）    （GitHub 上的契約範本）
        ↓                           ↓
        └──────── 自動下載並填充 ────┘
                      ↓
            📄 filled_contract.txt
              （已填入當事人的完整契約）
                      ↓
          🤖 呼叫 Claude API（LLM）
          依照 5 條審查規則逐一分析
                      ↓
            📊 輸出審查分析報告
```

---

### 📋 本教材使用的 5 條審查規則

| 編號 | 審查重點 | 說明 |
|------|---------|------|
| 規則1 | 付款條件公平性 | 頭期款比例是否超過30%？付款期限是否合理？ |
| 規則2 | 違約金是否過重 | 逾期違約金比例是否超過千分之5？ |
| 規則3 | 保密期間合理性 | 保密義務期間是否超過5年？ |
| 規則4 | 智慧財產權歸屬 | 是否有一方完全壟斷所有IP權利？ |
| 規則5 | 終止條款平衡性 | 是否只有單方（甲方）才能終止契約？ |

---

> ⚠️ **使用前請確認**：你需要一組 Anthropic Claude API 金鑰才能執行審查步驟。  
> 申請網址：https://console.anthropic.com/


## Step 1｜安裝必要套件

**這格在做什麼**：安裝呼叫 Claude AI 所需的套件。在 Colab 每次開啟都需執行一次。

In [ ]:
!pip install anthropic -q

print("✅ 套件安裝完成！")

## Step 2｜填入 Claude API 金鑰

**這格在做什麼**：把你的 API 金鑰存入程式，後續步驟會用它來呼叫 AI。

**如何取得金鑰：**
1. 前往 👉 https://console.anthropic.com/ 註冊帳號
2. 點選左側「API Keys」→「Create Key」
3. 複製金鑰（格式：`sk-ant-api03-...`）貼到下方

> ⚠️ 金鑰只會顯示一次，請立即複製並妥善保存！


In [ ]:
# 請將下方引號內的文字，替換成你自己的 Claude API 金鑰
CLAUDE_API_KEY = "sk-ant-api03-請填入你的Claude金鑰"

print("✅ 金鑰設定完成！")
print(f"   金鑰開頭：{CLAUDE_API_KEY[:20]}...")

## Step 3｜載入套件與初始化

**這格在做什麼**：載入所有需要的工具，並建立與 Claude AI 的連線。

In [ ]:
import json
import re
import requests
import anthropic
from datetime import datetime

BASE_URL = "https://raw.githubusercontent.com/mjib007/contract-review/main"

client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

print("✅ 所有套件載入完成！")
print(f"   Anthropic 版本：{anthropic.__version__}")
print(f"   執行時間：{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}")
print(f"   資料來源：{BASE_URL}")

## Step 4｜從 GitHub 下載契約範本與當事人資料

**這格在做什麼**：自動從 GitHub 下載兩個檔案：
- `contract_template.txt`：契約範本（含佔位符）
- `party_data.json`：當事人基本資料

> 💡 你可以直接修改 GitHub 上的 `party_data.json`，改成你自己的測試情境！


In [ ]:
# 下載契約範本
r1 = requests.get(f"{BASE_URL}/contract_template.txt")
r1.encoding = "utf-8"
template = r1.text
print(f"✅ 契約範本下載成功（{len(template)} 字元）")

# 下載當事人資料
r2 = requests.get(f"{BASE_URL}/party_data.json")
party_data = r2.json()
print(f"✅ 當事人資料下載成功")
print()
print(f"甲方：{party_data['PARTY_A_NAME']}（代表人：{party_data['PARTY_A_REPRESENTATIVE']}）")
print(f"乙方：{party_data['PARTY_B_NAME']}（代表人：{party_data['PARTY_B_REPRESENTATIVE']}）")
print()
print(f"契約總價：NT$ {party_data['TOTAL_AMOUNT']} 元")
print(f"頭期款：NT$ {party_data['DEPOSIT_AMOUNT']} 元")
print(f"違約金比例：千分之 {party_data['PENALTY_RATIO']}")
print(f"保密期間：{party_data['CONFIDENTIALITY_YEARS']} 年")

## Step 5｜自動填充契約範本

**這格在做什麼**：把所有 `{{佔位符}}` 替換成 JSON 資料中對應的真實資料，產生完整契約文字。

In [ ]:
filled_contract = template
for key, value in party_data.items():
    placeholder = "{{" + key + "}}"
    filled_contract = filled_contract.replace(placeholder, str(value))

remaining = re.findall(r'\{\{.*?\}\}', filled_contract)
if remaining:
    print(f"⚠️  注意：以下佔位符未被填充：{remaining}")
else:
    print("✅ 所有佔位符填充完成，無遺漏！")

print()
print("📄 已填充契約預覽（前400字）：")
print("─" * 50)
print(filled_contract[:400])
print("─" * 50)

## Step 6｜定義審查規則與 AI 審查函式

**這格在做什麼**：設定 B 公司的 5 條審查規則，以及送交 AI 審查的函式。

> 💡 修改下方 `REVIEW_RULES` 的內容，就能換成你自己設計的審查規則！


In [ ]:
REVIEW_RULES = [
    {
        "id": 1,
        "name": "付款條件公平性",
        "description": "頭期款比例是否超過契約總金額的30%？付款期限是否超過60個工作日？如有異常，請指出條文位置與風險。"
    },
    {
        "id": 2,
        "name": "違約金是否過重",
        "description": "每日逾期違約金的比例是否超過契約總價款的千分之5？過高的違約金對乙方（B公司）不利，請評估風險。"
    },
    {
        "id": 3,
        "name": "保密期間合理性",
        "description": "保密義務的期間是否超過5年？超過5年的保密義務在實務上可能過於嚴苛，請指出並提供修改建議。"
    },
    {
        "id": 4,
        "name": "智慧財產權歸屬",
        "description": "智慧財產權是否完全歸甲方所有，乙方是否毫無保留任何權利？請分析並建議修改方向。"
    },
    {
        "id": 5,
        "name": "終止條款平衡性",
        "description": "契約終止權是否只賦予甲方（A公司），乙方是否有對等的終止權？若雙方權利不對等，請指出並建議修改方式。"
    }
]

def review_contract_with_ai(contract_text, rule):
    prompt = f"""你是一位專業的企業法務顧問，正在協助 B 公司審查一份由 A 公司提供的技術服務契約。

請依照以下審查規則，仔細分析契約內容：

【審查規則 {rule['id']}：{rule['name']}】
{rule['description']}

【待審查契約全文】
{contract_text}

請以以下格式回應：
1. **審查結果**：✅ 無問題 / ⚠️ 有疑慮 / ❌ 建議修改
2. **問題條文**：指出有問題的條文編號與內容（若無問題則填「無」）
3. **風險說明**：說明為何構成問題、對 B 公司的潛在風險
4. **修改建議**：具體建議如何修改條文內容（若無問題則填「無需修改」）

請用繁體中文回應，語氣專業但易懂。"""

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

print("✅ 審查規則設定完成！")
for rule in REVIEW_RULES:
    print(f"   規則{rule['id']}：{rule['name']}")
print()
print("✅ AI 審查函式定義完成，準備開始審查！")

## Step 7｜執行自動審查

**這格在做什麼**：依序將契約送交 AI，按照 5 條規則逐一審查，每條規則完成後即時顯示結果。

> ⚠️ 這個步驟會呼叫 Claude API（約需 30–60 秒），請耐心等候。


In [ ]:
print("🔍 開始進行契約自動審查...")
print("=" * 60)
print()

review_results = []

for rule in REVIEW_RULES:
    print(f"📌 正在執行 規則{rule['id']}：{rule['name']}...")
    result = review_contract_with_ai(filled_contract, rule)
    review_results.append({"rule_id": rule['id'], "rule_name": rule['name'], "result": result})
    print(f"\n{'-' * 50}")
    print(f"【規則{rule['id']}：{rule['name']}】審查結果")
    print('-' * 50)
    print(result)
    print()

print("=" * 60)
print(f"✅ 全部 {len(REVIEW_RULES)} 條規則審查完成！")

## Step 8｜輸出完整審查報告

**這格在做什麼**：將所有審查結果整合成一份正式報告，並儲存為文字檔（可從 Colab 左側檔案區下載）。

In [ ]:
lines = []
lines.append("=" * 60)
lines.append("        契約條款審查分析報告")
lines.append("=" * 60)
lines.append(f"審查日期：{datetime.now().strftime("%Y年%m月%d日 %H:%M")}")
lines.append(f"甲方：{party_data['PARTY_A_NAME']}")
lines.append(f"乙方：{party_data['PARTY_B_NAME']}")
lines.append(f"契約標的：{party_data['SERVICE_DESCRIPTION']}")
lines.append("─" * 60)
lines.append("")

for item in review_results:
    lines.append(f"【規則{item['rule_id']}：{item['rule_name']}】")
    lines.append(item["result"])
    lines.append("")
    lines.append("─" * 60)
    lines.append("")

lines.append("【審查說明】")
lines.append("本報告由 AI 輔助生成，僅供參考。")
lines.append("最終契約決策仍應由具備法律專業資格之人員確認。")
lines.append("=" * 60)

report_text = "\n".join(lines)
print(report_text)

filename = f"contract_review_report_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"\n💾 報告已儲存為：{filename}")
print("📥 請到左側檔案區（資料夾圖示）下載此報告檔案")
print("✅ 完整流程執行完畢！")

## 🎓 延伸練習

恭喜完成本教材！以下是三個進階挑戰：

---

### 🔹 練習一：修改當事人資料（改 JSON）
前往 GitHub 上的 `party_data.json`，把違約金比例改成 `"8"`（千分之八），重新執行 Step 7～8，觀察 AI 的審查結果有何不同。

---

### 🔹 練習二：新增第六條審查規則
在 Step 6 的 `REVIEW_RULES` 列表中，仿照現有規則格式新增第六條：
- 規則名稱：「爭議解決機制」
- 審查重點：管轄法院是否對乙方方便？是否應加入仲裁條款？

---

### 🔹 練習三：改用不同 AI 模型
在 Step 6 的 `review_contract_with_ai` 函式中，把 `model` 參數改成 `"claude-haiku-4-5-20251001"`（較快速、費用較低），比較審查品質的差異。

---

> 💡 **思考問題**：AI 審查契約有哪些優點和限制？在實際法律工作中，AI 輔助審查應該扮演什麼角色？
